# Vegetation Index Crop Insurance (VICI)

VICI is an NDVI-based drought monitoring service designed for micro-insurance purposes.

This notebook guides you through all the steps required to prepare and run the VICI workflow.

The VICI product has been designed by prof. Kees de Bie at (ITC, University of Twente). VITO has converted this workflow into an operational service on Terrascope's EOPlaza platform for Ethiopia. 
More information: https://portal.terrascope.be/catalogue/app-details/186

In [ ]:
# TEMPORARY CELL
from pathlib import Path

outdir = Path('./results/vici/run_20250820_163224')
aoi_gpkg_file = outdir / "AOI.gpkg"

In [ ]:
# Set output directory for your run
from pathlib import Path
from datetime import datetime

now = datetime.now().strftime('%Y%m%d_%H%M%S')
outdir = Path(f'./results/vici/run_{now}')
outdir.mkdir(parents=True, exist_ok=True)

### Draw your region of interest

In [ ]:
from vito_agri_tutorials.utils.map import ui_map

map = ui_map()

In [ ]:
# Get the AOI from the map and save as GeoPackage file
aoi_gdf = map.get_objects()

# Save to geopackage file
aoi_gpkg_file = outdir / "AOI.gpkg"
aoi_gdf.to_file(aoi_gpkg_file, driver="GPKG")

### Preparations: process 20 year NDVI archive

#### Step 1: Define archive period

In [ ]:
from vito_agri_tutorials.vici import get_vici_archive_dates

# Specify final year of 20-year reference period
end_year_archive = 2019

start_date_archive, end_date_archive = get_vici_archive_dates(end_year_archive)

#### Step 2: Gather required NDVI imagery

Note that we automatically extend the time series with 6 dekads (2 months) before and 6 dekads after the requested archiving period to facilitate smoothing of the time series in a later stage.

In [ ]:
from vito_agri_tutorials.vici import get_ndvi_data_terrascope

archive_dir = outdir / "NDVI_archive"
ndvi_archive_dir = archive_dir / "NDVI_original"
get_ndvi_data_terrascope(
        aoi_gpkg_file, ndvi_archive_dir, start_date_archive, end_date_archive
    )

Let's inspect an individual NDVI image and a time series for one pixel!

In [ ]:
import glob
from matplotlib import pyplot as plt
from vito_agri_tutorials.utils.geotiff import read_geotiff
from vito_agri_tutorials.vici import NDVI_SCALE, NDVI_OFFSET

# Get the files
infiles = sorted(glob.glob(str(ndvi_archive_dir / "*.tif")))
infile = infiles[0]
print('File to load:')
print(infile)

# Read the file
ndvi_data = read_geotiff(infile)
print('Range of loaded data:', ndvi_data.min(), ndvi_data.max())

# rescale the NDVI to meaningful values
ndvi_data = (ndvi_data * NDVI_SCALE) + NDVI_OFFSET
print('Range of rescaled data:', ndvi_data.min(), ndvi_data.max())

# Visualize the image
plt.imshow(ndvi_data, cmap='RdYlGn')
plt.colorbar(label='NDVI')
plt.title('Normalized Difference Vegetation Index (NDVI)')
plt.show()


In [ ]:
import numpy as np

# Now load timeseries for one pixel and one year
ndvi_ori = []
infiles = sorted(glob.glob(str(ndvi_archive_dir / "*.tif")))
files_to_load = infiles[0:108]  # First 3 years (108 dekads)
for file_path in files_to_load:
    ndvi  = read_geotiff(file_path)
    ndvi_ori.append(ndvi)
ndvi_ori = np.array(ndvi_ori)

# Extract values for one pixel
pixel_ori = ndvi_ori[:, 5, 5]

# Convert to meaningful values
pixel_ori = (pixel_ori * NDVI_SCALE) + NDVI_OFFSET

# Plot the timeseries
fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(pixel_ori, '-b', label='Original NDVI')
ax.set_xlabel('Dekad number')
ax.set_ylabel('NDVI')
ax.legend()

#### Step 3: Apply upper envelope smoothing

Note that whenever a pixel contains a consecutive sequence of more than 12 dekads with no data, the entire pixel becomes invalid!


In [ ]:
from vito_agri_tutorials.vici import upper_envelope_smoothing

ndvi_smoothed_file = archive_dir / "NDVI_smoothed.tif"
upper_envelope_smoothing(
        ndvi_archive_dir, start_date_archive, end_date_archive, ndvi_smoothed_file
    )

In [ ]:
from vito_agri_tutorials.utils.geotiff import read_geotiff
from matplotlib import pyplot as plt
import glob
import numpy as np

# Now visualize the result of the smoothing procedure

# Get original NDVI data from separate files
ndvi_ori = []
infiles = sorted(glob.glob(str(ndvi_archive_dir / "*.tif")))
for file_path in infiles:
    ndvi  = read_geotiff(file_path)
    ndvi_ori.append(ndvi)
ndvi_ori = np.array(ndvi_ori)

# Extract values for one pixel
pixel_ori = ndvi_ori[:, 5, 5]
pixel_ori = (pixel_ori * NDVI_SCALE) + NDVI_OFFSET

# Get smoothed NDVI data
smoothed = read_geotiff(ndvi_smoothed_file)

# extract the same pixel
pixel_smoothed = smoothed[:, 5, 5]
pixel_smoothed = (pixel_smoothed * NDVI_SCALE) + NDVI_OFFSET

# Extend the smoothed pixel timeseries with 6 dekads before and 6 dekads after
pixel_smoothed = np.concatenate((np.array(np.repeat(np.nan, 6)), pixel_smoothed, np.array(np.repeat(np.nan, 6))))

fig, ax = plt.subplots(figsize=(10, 10))
ax.plot(pixel_ori, '-b', label='Original NDVI')
ax.plot(pixel_smoothed, '-r', label='Smoothed NDVI')
ax.set_xlabel('Dekad number')
ax.set_ylabel('NDVI')
ax.legend()

#### Step 4: Define invalid pixels

Now we determine for each pixel, based on the available NDVI data (20 years), whether or not we will include that pixel in our analysis.

We apply two checks:
1. If 

In [ ]:
from vito_agri_tutorials.vici import create_invalid_pixel_mask

invalid_pixel_mask_file = archive_dir / "invalid_pixel_mask.tif"

invalid_pixel_mask = create_invalid_pixel_mask(
    ndvi_smoothed_file, invalid_pixel_mask_file
)

#### Step 5: Compute long-term statistics for each dekad

In [ ]:
from vito_agri_tutorials.vici import compute_stats_per_dekad

stats_per_dekad_file = archive_dir / "stats_per_dekad.tif"
stats_per_dekad = compute_stats_per_dekad(
        ndvi_smoothed_file, invalid_pixel_mask_file, stats_per_dekad_file
    )

#### Step 6: Define Crop Production Zones (CPSZs)

In [ ]:
from vito_agri_tutorials.vici import determine_clusters_kmeans

cpsz_file = archive_dir / "cpsz.tif"

determine_clusters_kmeans(
        stats_per_dekad_file,
        cpsz_file,
        min_zones=2,
        max_zones=5,
        sub_sample=10,
    )

#### Step 7: Compute NDVI adjustment factor per zone and correct NDVI archive

In [ ]:
from vito_agri_tutorials.vici import compute_zonal_ndvi_adjustments, apply_zonal_ndvi_adjustments

zonal_adjustments_file = archive_dir / "zonal_adjustments.tif"
zonal_adjustments_dekad_dir = archive_dir / "zonal_adjustments_dekad"

compute_zonal_ndvi_adjustments(
    ndvi_smoothed_file,
    cpsz_file,
    zonal_adjustments_file,
    zonal_adjustments_dekad_dir,
)

ndvi_adjusted_file = archive_dir / "NDVI_adjusted.tif"

apply_zonal_ndvi_adjustments(
    ndvi_smoothed_file, zonal_adjustments_file, ndvi_adjusted_file
)

#### Step 8: Compute payout thresholds

In [ ]:
from vito_agri_tutorials.vici import compute_payout_thresholds

final_thresholds_dir = archive_dir / "final_thresholds"

percentiles, p50_array = compute_payout_thresholds(
        ndvi_adjusted_file, cpsz_file, final_thresholds_dir
    )

#### Step 9: Derive growing seasons per zone

In [ ]:
from vito_agri_tutorials.vici import define_growing_seasons

outdir_seasons = archive_dir / "seasons"

seasons = define_growing_seasons(p50_array, cpsz_file, outdir_seasons)

In [ ]:
# Visualize growing seasons

xxx

### Compute VICI

In [ ]:
from vito_agri_tutorials.vici import run_vici

# Define start and end date of interest
start_date = "2021-01-01"
end_date = "2021-12-21"

# Run the VICI processing
vici, quality_flags = run_vici(outdir, start_date, end_date)

In [ ]:
# Report on results

## Summarize quality flags??

## Show timeseries of VICI raster values??

# Statistics on VICI rasters